# 부두 선석별 월 대기시간 및 재항시간

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def load_dataset(filename):
    BASE_DIR = './data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'
    try:
        return pd.read_csv(TARGET_PATH)
    except FileNotFoundError:
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding = 'euc-kr')


port1, port2, port3, port4, port5 = [
    load_dataset(f'port{i}') for i in range(1, 6)
]

record = load_dataset("record")

import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")

def read_csv_file(filename):
    path = DATA_DIR / filename

    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp949")


# port1~5 합치기
port_list = []

for i in range(1, 6):
    p = read_csv_file(f"port{i}.csv")

    # port3만 '선박명'이라는 열 이름을 사용
    p = p.rename(columns={
        "선박명": "선명",
        "선석": "접안자료선석"
    })

    p["부두"] = i
    port_list.append(p)

port = pd.concat(port_list, ignore_index=True)
port = port.drop_duplicates().copy()
port["접안ID"] = range(len(port))

rec = read_csv_file("record.csv").drop_duplicates().copy()


# 날짜 변환
port["접안시각"] = pd.to_datetime(
    port["접안(예정)일시"], format="mixed", errors="coerce"
)
port["선석종료시각"] = pd.to_datetime(
    port["출항(예정)일시"], format="mixed", errors="coerce"
)

rec["입항시각"] = pd.to_datetime(
    rec["입항일시"], format="mixed", errors="coerce"
)
rec["출항시각"] = pd.to_datetime(
    rec["출항일시"], format="mixed", errors="coerce"
)


# record의 계선장소에서 부두·선석 추출
rec[["부두", "선석"]] = (
    rec["계선장소"]
    .str.extract(r"신항\s*([1-5])\s*부두\s*(\d+)\s*선석")
    .apply(pd.to_numeric)
)


# 선명 공백·대소문자 통일
for df in [port, rec]:
    df["연결선명"] = (
        df["선명"].str.upper()
        .str.replace(r"\s+", "", regex=True)
    )


# 필요한 정보가 없는 기록 제외
port = port.dropna(
    subset=["연결선명", "접안시각", "선석종료시각"]
).copy()

rec = rec.dropna(
    subset=["부두", "선석", "연결선명", "입항시각", "출항시각"]
).copy()

port = port[
    port["선석종료시각"] >= port["접안시각"]
].copy()

rec = rec[
    (rec["출항시각"] >= rec["입항시각"])
    & (rec["입항시각"] >= "2022-01-01")
    & (rec["입항시각"] < "2026-01-01")
].copy()

rec[["부두", "선석"]] = rec[["부두", "선석"]].astype(int)

rec["재항시간"] = (
    rec["출항시각"] - rec["입항시각"]
).dt.total_seconds() / 3600

rec["월"] = rec["입항시각"].dt.to_period("M").astype(str)

ships = pd.merge_asof(
    rec.sort_values("출항시각"),

    port[[
        "부두", "연결선명", "접안자료선석",
        "접안시각", "선석종료시각", "접안ID"
    ]].sort_values("선석종료시각"),

    left_on="출항시각",
    right_on="선석종료시각",
    by=["부두", "연결선명"],
    direction="nearest",
    tolerance=pd.Timedelta(hours=24)
)

ships["출항시각차이"] = (
    ships["출항시각"] - ships["선석종료시각"]
).abs()


# 입항 이후, 출항 이전에 접안한 기록만 사용
valid = ships[
    (ships["접안시각"] >= ships["입항시각"])
    & (ships["접안시각"] <= ships["출항시각"])
].copy()


# 같은 접안 기록을 여러 입항에 중복 사용하지 않음
valid = (
    valid.sort_values("출항시각차이")
    .drop_duplicates("접안ID")
    .copy()
)

valid["대기시간"] = (
    valid["접안시각"] - valid["입항시각"]
).dt.total_seconds() / 3600

group_cols = ["부두", "선석", "월"]

# record 전체 기준
monthly_all = (
    rec.groupby(group_cols)
    .agg(
        전체입항건수=("재항시간", "size"),
        전체_평균재항시간=("재항시간", "mean")
    )
)

# 접안 기록까지 연결된 표본 기준
monthly_matched = (
    valid.groupby(group_cols)
    .agg(
        대기시간계산건수=("대기시간", "size"),
        평균대기시간=("대기시간", "mean"),
        연결표본_평균재항시간=("재항시간", "mean")
    )
)

monthly = monthly_all.join(monthly_matched)

monthly["대기시간계산건수"] = (
    monthly["대기시간계산건수"].fillna(0).astype(int)
)

monthly["분석포함률(%)"] = (
    monthly["대기시간계산건수"]
    / monthly["전체입항건수"] * 100
)

monthly = monthly.reset_index().sort_values(group_cols)

display(monthly.round(2))

# 체류생산성

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

# 1. 분석 기간과 원본 컬럼 설정
DATA_PATH = 'C:/final/data'
START_YEAR, END_YEAR = 2022, 2025

BERTH_COL = '선석'
ARRIVAL_COL = '접안(예정)일시'
DEPARTURE_COL = '출항(예정)일시'
HANDLING_COLS = ['양하', '적하', 'Shift']
YEAR_COL, MONTH_COL = '기준년도', '월'
PORT_MONTH_COLS = [f'신항{i}부두' for i in range(1, 6)]

# 2. 원본 데이터 불러오기
port1 = pd.read_csv(f'{DATA_PATH}/port1.csv', encoding='utf-8-sig')
port2 = pd.read_csv(f'{DATA_PATH}/port2.csv', encoding='cp949')
port3 = pd.read_csv(f'{DATA_PATH}/port3.csv', encoding='cp949')
port4 = pd.read_csv(f'{DATA_PATH}/port4.csv', encoding='cp949')
port5 = pd.read_csv(f'{DATA_PATH}/port5.csv', encoding='cp949')
port_month = pd.read_csv(f'{DATA_PATH}/port_month.csv', encoding='cp949')

# 3. 원본을 복사한 뒤 날짜, 물동량, 재항시간을 계산하고 이상값 제외
def prepare_port_data(data, port):
    data = data.copy()
    # 원본 데이터는 수정하지 않고 복사본에서 날짜를 변환
    data[ARRIVAL_COL] = pd.to_datetime(data[ARRIVAL_COL], errors='coerce')
    data[DEPARTURE_COL] = pd.to_datetime(data[DEPARTURE_COL], errors='coerce')
    # 양하 + 적하 + Shift를 계산하기 위해 물동량 컬럼을 숫자로 변환
    data[HANDLING_COLS] = data[HANDLING_COLS].apply(
        lambda column: pd.to_numeric(column.astype(str).str.replace(',', '', regex=False), errors='coerce')
    )
    # 선박별 처리 TEU = 양하 + 적하 + Shift
    data['processing_teu'] = data[HANDLING_COLS].sum(axis=1, min_count=len(HANDLING_COLS))
    # 선박별 재항시간 = 출항(예정)일시 - 접안(예정)일시
    data['stay_hours'] = (data[DEPARTURE_COL] - data[ARRIVAL_COL]).dt.total_seconds() / 3600
    data['year_month'] = data[ARRIVAL_COL].dt.to_period('M').astype(str)
    data['date'] = data[ARRIVAL_COL].dt.to_period('M').dt.to_timestamp()
    data['port'] = port
    data['berth'] = data[BERTH_COL].astype(str)
    # 분석 기간과 이상값 제외 기준을 적용
    valid = (
        data[ARRIVAL_COL].dt.year.between(START_YEAR, END_YEAR)
        & data[HANDLING_COLS].ge(0).all(axis=1)
        & data['processing_teu'].gt(0)
        & data['stay_hours'].gt(0)
        & ~data.duplicated()
    )
    return data.loc[valid].copy()

# 4. 부두별 복사본을 결합해 분석용 전체 데이터 생성
raw_port_data = {'port1': port1, 'port2': port2, 'port3': port3, 'port4': port4, 'port5': port5}
all_data = pd.concat(
    [prepare_port_data(data, port) for port, data in raw_port_data.items()],
    ignore_index=True,
)

# 5. 월별·부두별·선석별 물동량과 재항시간 합계 계산
berth_result = (
    all_data.groupby(['port', 'year_month', 'date', 'berth'], as_index=False)
    .agg(berth_volume=('processing_teu', 'sum'), stay_hours=('stay_hours', 'sum'))
)
# 같은 부두·월의 전체 물동량을 구해 선석별 배분 비율 계산
berth_result['port_volume'] = berth_result.groupby(['port', 'year_month'])['berth_volume'].transform('sum')
berth_result['berth_share'] = berth_result['berth_volume'] / berth_result['port_volume'].replace(0, np.nan)

# 6. 부두별 월간 천TEU 데이터를 선석별 배분에 사용할 수 있도록 정리
port_month_work = port_month.copy()
port_month_work[YEAR_COL] = pd.to_numeric(port_month_work[YEAR_COL], errors='coerce')
port_month_work[MONTH_COL] = pd.to_numeric(port_month_work[MONTH_COL], errors='coerce')
port_month_work = port_month_work[port_month_work[YEAR_COL].between(START_YEAR, END_YEAR)].copy()
port_month_work['year_month'] = (
    port_month_work[YEAR_COL].astype(int).astype(str)
    + '-' + port_month_work[MONTH_COL].astype(int).astype(str).str.zfill(2)
)
port_month_long = port_month_work.melt(
    id_vars='year_month', value_vars=PORT_MONTH_COLS,
    var_name='source_port', value_name='port_thousand_teu'
)
port_month_long['port'] = port_month_long['source_port'].map(
    {column: f'port{i}' for i, column in enumerate(PORT_MONTH_COLS, 1)}
)
port_month_long['port_thousand_teu'] = pd.to_numeric(port_month_long['port_thousand_teu'], errors='coerce')

# 7. 부두별 천TEU를 선석별 물동량 비율로 배분
berth_result = berth_result.merge(
    port_month_long[['port', 'year_month', 'port_thousand_teu']],
    on=['port', 'year_month'], how='left'
)
# 선석별 추정 천TEU = 부두별 월간 천TEU × 선석별 배분 비율
berth_result['estimated_thousand_teu'] = berth_result['port_thousand_teu'] * berth_result['berth_share']
# 선석별 체류생산성 = 선석별 추정 천TEU ÷ 선석별 재항시간
berth_result['stay_productivity'] = berth_result['estimated_thousand_teu'] / berth_result['stay_hours'].replace(0, np.nan)

# 8. 부두별 그래프를 위해 선석별 체류생산성의 평균 계산
port_result = (
    berth_result.groupby(['port', 'year_month'], as_index=False)
    .agg(stay_productivity=('stay_productivity', 'mean'), berth_count=('berth', 'nunique'))
)
port_result['date'] = pd.to_datetime(port_result['year_month'])

# 9. 월별·부두별 평균 체류생산성 Plotly 그래프 생성
fig = px.line(
    port_result, x='date', y='stay_productivity', color='port', markers=True,
    hover_data={'date': False, 'year_month': True, 'stay_productivity': ':.4f', 'berth_count': True}
)
fig.update_layout(
    title='월별·부두별 평균 체류생산성',
    xaxis_title='년월', yaxis_title='선석별 평균 체류생산성 (천TEU/시간)',
    hovermode='x unified', template='plotly_white', legend_title='부두'
)
fig.update_xaxes(dtick='M3', tickformat='%Y-%m', tickangle=-45)
fig.show()

# 10. 부두별 및 선석별 계산 결과 목록 출력
port_table = port_result.rename(columns={
    'port': '부두', 'year_month': '년월', 'berth_count': '선석수',
    'stay_productivity': '체류생산성 (천TEU/시간)'
})[['부두', '년월', '선석수', '체류생산성 (천TEU/시간)']]

berth_table = berth_result.rename(columns={
    'port': '부두', 'year_month': '년월', 'berth': '선석',
    'berth_share': '선석별배분비율', 'estimated_thousand_teu': '선석별천TEU',
    'stay_hours': '재항시간(시간)', 'stay_productivity': '체류생산성 (천TEU/시간)'
})[['부두', '년월', '선석', '선석별배분비율', '선석별천TEU', '재항시간(시간)', '체류생산성 (천TEU/시간)']]

print('부두별 결과 목록')
display(port_table)
print('선석별 결과 목록')
display(berth_table)

# 선석점유율